# Week 3: Brownian Motion II: Itô Intuition and Geometric Brownian Motion

**Project:** *Valuing Variable Annuity Guarantees Under Market Crashes and Regime Shifts*  
**Mentor:** Hao Quan  
**Notebook type:** all-in-one lesson, lab, exercises, and worked solutions

> **Driving question:** How does Brownian motion become a positive account-value model, and where does the Itô correction come from?

## Learning goals

- Interpret drift and volatility in a stochastic differential equation.
- Connect Brownian quadratic variation to the Itô correction.
- Derive and simulate the exact geometric Brownian-motion solution.
- Compare exact GBM with Euler–Maruyama using the same shocks.
- Distinguish physical-path simulation from risk-neutral valuation.

## How to use this notebook

1. Complete the assigned reading before the weekly meeting.
2. Read the explanation cells and predict each result before running the code.
3. Run the notebook from top to bottom. Every random experiment uses a fixed seed.
4. Attempt the exercises before opening the worked-solution section.
5. Write a 150–250 word interpretation of the main result in your own words.

This notebook is educational. It simplifies real contracts and is not an insurance
quotation, investment recommendation, or complete actuarial valuation.

## Assigned reading

- [MIT OCW Stochastic Calculus lecture](https://ocw.mit.edu/courses/18-642-topics-in-mathematics-with-applications-in-finance-fall-2024/mit18_642_f24_lec19_1.pdf) — focus on Brownian motion with drift, Itô's formula, and the GBM slides
- [MIT OCW Lecture 17](https://ocw.mit.edu/courses/18-s096-topics-in-mathematics-with-applications-in-finance-fall-2013/3b97c6b0c282dd9dc024c4c7ffe3fba8_MIT18_S096F13_lecnote17.pdf) — review the Brownian definition and quadratic-variation discussion


In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.set_printoptions(precision=4, suppress=True)

SEED = 20260730
rng = np.random.default_rng(SEED)
print(f"Reproducible random seed: {SEED}")


## 1. From arithmetic changes to percentage changes

An arithmetic Brownian model can become negative. A standard financial model
instead makes **percentage changes** random:

$$dS_t=\mu S_t\,dt+\sigma S_t\,dW_t.$$

- $\mu$ is the expected continuously compounded return under the selected
  probability measure.
- $\sigma$ is annualized volatility.
- The units of time must match the annualized parameters.

Applying Itô's formula to $\log S_t$ gives

$$d\log S_t=\left(\mu-\frac12\sigma^2\right)dt+\sigma dW_t,$$

and therefore

$$S_t=S_0\exp\left[
\left(\mu-\frac12\sigma^2\right)t+\sigma W_t
\right].$$


## 2. Why ordinary calculus misses a term

For a small Brownian increment, $\Delta W=O(\sqrt{\Delta t})$, so
$(\Delta W)^2=O(\Delta t)$ and cannot be discarded. The experiment below
revisits quadratic variation and contrasts it with a smooth path.


In [ ]:
def quadratic_variation(values):
    values = np.asarray(values)
    return np.sum(np.diff(values) ** 2)


rows = []
for n_steps in [32, 128, 512, 2048]:
    local_rng = np.random.default_rng(SEED + n_steps)
    dt = 1 / n_steps
    brownian = np.concatenate([[0.0], np.cumsum(np.sqrt(dt) * local_rng.standard_normal(n_steps))])
    t = np.linspace(0, 1, n_steps + 1)
    smooth = np.sin(2 * np.pi * t)
    rows.append([n_steps, quadratic_variation(brownian), quadratic_variation(smooth)])

qv_compare = pd.DataFrame(rows, columns=["steps", "Brownian_QV", "smooth_QV"])
print(qv_compare.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(qv_compare["steps"], qv_compare["Brownian_QV"], marker="o", label="Brownian")
ax.semilogx(qv_compare["steps"], qv_compare["smooth_QV"], marker="o", label="Smooth sine")
ax.axhline(1, color="black", linestyle="--", linewidth=1, label="Brownian target T=1")
ax.set(title="Quadratic variation separates rough and smooth paths", xlabel="Grid steps", ylabel="Sum of squared increments")
ax.legend()
plt.show()


## 3. Exact GBM simulator

For a grid step $\Delta t$,

$$S_{t+\Delta t}=S_t\exp\left[
(\mu-\tfrac12\sigma^2)\Delta t
+\sigma\sqrt{\Delta t}Z
\right].$$


In [ ]:
def gbm_exact_paths(S0, mu, sigma, T, n_steps, n_paths, seed=SEED, normals=None):
    if S0 <= 0 or sigma < 0 or T <= 0:
        raise ValueError("S0 and T must be positive; sigma must be nonnegative")
    dt = T / n_steps
    if normals is None:
        generator = np.random.default_rng(seed)
        normals = generator.standard_normal((n_paths, n_steps))
    normals = np.asarray(normals)
    if normals.shape != (n_paths, n_steps):
        raise ValueError("normals have the wrong shape")
    log_increments = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * normals
    log_paths = np.column_stack([np.zeros(n_paths), np.cumsum(log_increments, axis=1)])
    paths = S0 * np.exp(log_paths)
    times = np.linspace(0, T, n_steps + 1)
    return times, paths


t, paths = gbm_exact_paths(
    S0=100, mu=0.07, sigma=0.20, T=5, n_steps=5 * 12, n_paths=20, seed=SEED + 1
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, paths.T, linewidth=1)
ax.set(title="Exact geometric Brownian-motion paths", xlabel="Years", ylabel="Value")
plt.show()

assert (paths > 0).all()


## 4. The $-\sigma^2/2$ correction is not optional

The correct exact solution has mean $E[S_T]=S_0e^{\mu T}$. If the correction
is omitted, the mean is multiplied by $e^{\sigma^2T/2}$ and is biased upward.


In [ ]:
mean_rng = np.random.default_rng(SEED + 2)
n = 250_000
S0, mu, sigma, T = 100.0, 0.07, 0.25, 3.0
Z = mean_rng.standard_normal(n)

correct = S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
naive = S0 * np.exp(mu * T + sigma * np.sqrt(T) * Z)
target_mean = S0 * np.exp(mu * T)

mean_check = pd.DataFrame({
    "method": ["Correct Itô solution", "Naive formula", "Theory"],
    "mean_terminal_value": [correct.mean(), naive.mean(), target_mean],
})
print(mean_check.to_string(index=False))
print(f"Naive relative bias: {naive.mean() / target_mean - 1:.2%}")

mc_se = correct.std(ddof=1) / np.sqrt(n)
assert abs(correct.mean() - target_mean) < 4 * mc_se
assert naive.mean() > target_mean * 1.08


## 5. Exact simulation versus Euler–Maruyama

Euler uses the local linear update

$$S_{k+1}=S_k+\mu S_k\Delta t+\sigma S_k\sqrt{\Delta t}Z_{k+1}.$$

It is an approximation and can even become negative on a coarse grid. We use
the **same normal shocks** for exact and Euler paths so that the numerical
difference is not hidden by unrelated Monte Carlo noise.


In [ ]:
def gbm_euler_paths(S0, mu, sigma, T, normals):
    normals = np.asarray(normals)
    n_paths, n_steps = normals.shape
    dt = T / n_steps
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    for k in range(n_steps):
        paths[:, k + 1] = paths[:, k] * (
            1 + mu * dt + sigma * np.sqrt(dt) * normals[:, k]
        )
    return np.linspace(0, T, n_steps + 1), paths


compare_rng = np.random.default_rng(SEED + 3)
normals = compare_rng.standard_normal((8_000, 12))
_, exact_monthly = gbm_exact_paths(100, 0.06, 0.35, 1, 12, 8_000, normals=normals)
_, euler_monthly = gbm_euler_paths(100, 0.06, 0.35, 1, normals)

rmse_monthly = np.sqrt(np.mean((exact_monthly[:, -1] - euler_monthly[:, -1])**2))
print(f"Monthly-grid terminal RMSE: {rmse_monthly:.4f}")
print(f"Fraction of negative Euler values: {(euler_monthly <= 0).mean():.6f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
for i in range(3):
    ax.plot(np.linspace(0, 1, 13), exact_monthly[i], label=f"Exact path {i+1}")
    ax.plot(np.linspace(0, 1, 13), euler_monthly[i], linestyle="--", alpha=0.8)
ax.set(title="Exact (solid) and Euler (dashed) with identical shocks", xlabel="Years", ylabel="Value")
ax.legend(ncol=3, fontsize=8)
plt.show()


In [ ]:
step_rows = []
for n_steps in [4, 12, 52, 252]:
    step_rng = np.random.default_rng(SEED + 100 + n_steps)
    z = step_rng.standard_normal((15_000, n_steps))
    _, exact = gbm_exact_paths(100, 0.06, 0.35, 1, n_steps, 15_000, normals=z)
    _, euler = gbm_euler_paths(100, 0.06, 0.35, 1, z)
    rmse = np.sqrt(np.mean((exact[:, -1] - euler[:, -1])**2))
    step_rows.append([n_steps, 1/n_steps, rmse, (euler <= 0).mean()])

euler_check = pd.DataFrame(
    step_rows, columns=["steps", "dt", "terminal_RMSE", "negative_fraction"]
)
print(euler_check.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.loglog(euler_check["dt"], euler_check["terminal_RMSE"], marker="o")
ax.set(title="Euler terminal error shrinks on finer grids", xlabel="$\Delta t$", ylabel="RMSE")
plt.show()

assert euler_check.iloc[-1]["terminal_RMSE"] < euler_check.iloc[0]["terminal_RMSE"]


## 6. Add VA account charges

If the fund earns drift $\mu$ and charges are deducted continuously at rate
$c$, the simplified account follows

$$dA_t=(\mu-c)A_tdt+\sigma A_tdW_t.$$

Under a physical measure $P$, $\mu$ may represent an expected return used for
scenario analysis. Under a risk-neutral pricing measure $Q$, the pre-charge
drift becomes $r$, so the account drift is $r-c$.


In [ ]:
A0, r, physical_mu, charge, sigma, T = 100_000, 0.03, 0.075, 0.02, 0.18, 10
n_paths = 150_000
terminal_rng = np.random.default_rng(SEED + 4)
Z = terminal_rng.standard_normal(n_paths)

account_P = A0 * np.exp(
    (physical_mu - charge - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z
)
account_Q = A0 * np.exp(
    (r - charge - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z
)

comparison = pd.DataFrame({
    "measure": ["Physical P", "Pricing Q"],
    "drift_before_charge": [physical_mu, r],
    "simulated_mean": [account_P.mean(), account_Q.mean()],
    "theory_mean": [
        A0 * np.exp((physical_mu - charge) * T),
        A0 * np.exp((r - charge) * T),
    ],
    "p05": [np.quantile(account_P, 0.05), np.quantile(account_Q, 0.05)],
})
print(comparison.to_string(index=False, formatters={
    "simulated_mean": "${:,.2f}".format,
    "theory_mean": "${:,.2f}".format,
    "p05": "${:,.2f}".format,
}))

q_se = account_Q.std(ddof=1) / np.sqrt(n_paths)
assert abs(account_Q.mean() - A0 * np.exp((r - charge) * T)) < 4 * q_se


## 7. Common mistakes

- Writing `sigma = 20` instead of `sigma = 0.20`.
- Scaling a one-month shock by $1/12$ rather than $\sqrt{1/12}$.
- Confusing the arithmetic drift $\mu$ with the log drift
  $\mu-\sigma^2/2$.
- Using historical expected return $\mu$ inside a risk-neutral price.
- Calling Euler “exact” merely because the time step is small.
- Forgetting that GBM is continuous and cannot create an instantaneous crash.

## 8. Exercises

1. Verify the theoretical mean of a GBM with $S_0=80$, $\mu=5\%$,
   $\sigma=30\%$, and $T=2$.
2. Compare a fee-free account and an account with a 2% continuous charge.
3. Explain why $e^{-rt}S_t$ should have constant expectation under $Q$, while
   $e^{-rt}A_t$ declines when account charges are deducted.


## 9. Worked solutions


In [ ]:
# Exercise 1
exercise_rng = np.random.default_rng(SEED + 5)
z = exercise_rng.standard_normal(200_000)
S_T = 80 * np.exp((0.05 - 0.5 * 0.30**2) * 2 + 0.30 * np.sqrt(2) * z)
target = 80 * np.exp(0.05 * 2)
print(f"Simulated mean: {S_T.mean():.4f}; theory: {target:.4f}")

# Exercise 2
with_fee = 100 * np.exp((0.03 - 0.02 - 0.5 * 0.18**2) * 10 + 0.18 * np.sqrt(10) * z)
without_fee = 100 * np.exp((0.03 - 0.5 * 0.18**2) * 10 + 0.18 * np.sqrt(10) * z)
print(f"Mean without fee: {without_fee.mean():.2f}")
print(f"Mean with 2% fee: {with_fee.mean():.2f}")
print(f"Pathwise ratio with/without fee: {np.median(with_fee / without_fee):.4f}")

assert abs(S_T.mean() - target) < 4 * S_T.std(ddof=1) / np.sqrt(len(S_T))
assert np.allclose(with_fee / without_fee, np.exp(-0.02 * 10))


## 10. Weekly takeaways

1. Brownian quadratic variation makes second-order terms matter.
2. The Itô correction produces the correct log drift.
3. Exact GBM stays positive; Euler is an approximation.
4. Charges reduce the customer's account drift.
5. $P$ answers scenario/forecast questions; $Q$ is the valuation language
   introduced formally in Week 4.

### Weekly submission

- Completed notebook
- A short derivation of the GBM solution
- A paragraph distinguishing physical and risk-neutral simulation
